Buses - vibe coded

In [1]:
from google.colab import userdata

API_KEY = userdata.get('METLINK_API_KEY')\

API_KEY

'd7fySz2uP75LoRSnGFEVH6Tb1rpKNcvQ78YCKP5G'

In [2]:
import requests
import pandas as pd
from datetime import datetime, timedelta

# Metlink API configuration
HEADERS = {
    'accept': 'application/json',
    'x-api-key': API_KEY
}
BASE_URL = 'https://api.metlink.org.nz/v1'

# Key Stop IDs
WELLINGTON_STATION_STOPS = ['5000', '5506', '5012', '5006']
KELBURN_CAMPUS_STOP = '5416'

def get_departures(stop_id):
    url = f"{BASE_URL}/stop-predictions?stop_id={stop_id}"
    try:
        response = requests.get(url, headers=HEADERS, timeout=5)
        if response.status_code == 200:
            return response.json().get('predictions', [])
    except requests.exceptions.RequestException:
        # Network connection issue: return None to trigger simulated fallback
        return None
    return []

# Try to collect real-time departures
all_predictions = []
network_working = True

for stop in WELLINGTON_STATION_STOPS:
    predictions = get_departures(stop)
    if predictions is None:
        network_working = False
        break
    all_predictions.extend(predictions)

valid_buses = []

if not network_working:
    print("ℹ️ Network unreachable. Using simulated fallback data for routes 18, 21, and 22 to Kelburn...")
    # Generate realistic simulated upcoming buses to Kelburn Campus
    now = datetime.now()
    simulated_predictions = [
        {
            'route_id': '22',
            'destination': {'name': 'Mairangi via Kelburn'},
            'departure': {'scheduled': (now + timedelta(minutes=7)).isoformat()},
            'status': 'Delayed (2 mins)',
            'stop_id': '5006'
        },
        {
            'route_id': '18e',
            'destination': {'name': 'Karori South via Kelburn'},
            'departure': {'scheduled': (now + timedelta(minutes=14)).isoformat()},
            'status': 'On time',
            'stop_id': '5000'
        },
        {
            'route_id': '21',
            'destination': {'name': 'Karori (Landfill) via Kelburn'},
            'departure': {'scheduled': (now + timedelta(minutes=22)).isoformat()},
            'status': 'On time',
            'stop_id': '5012'
        }
    ]
    for pred in simulated_predictions:
        dep_time = datetime.fromisoformat(pred['departure']['scheduled'])
        valid_buses.append({
            'Route': pred['route_id'],
            'Destination': pred['destination']['name'],
            'Scheduled Departure': dep_time.strftime('%Y-%m-%d %H:%M:%S'),
            'Status': pred['status'],
            'Stop ID': pred['stop_id']
        })
else:
    for pred in all_predictions:
        departure_time_str = pred.get('departure', {}).get('scheduled') or pred.get('departure', {}).get('aimed')
        if not departure_time_str:
            continue
        try:
            dep_time = datetime.fromisoformat(departure_time_str.replace('Z', '+00:00'))
        except ValueError:
            continue

        route_id = pred.get('route_id', '')
        # Route 18, 21, 22 are known to pass through Kelburn Campus (Stop 5416)
        if any(r in route_id for r in ['18', '21', '22']):
            valid_buses.append({
                'Route': route_id,
                'Destination': pred.get('destination', {}).get('name', 'Unknown'),
                'Scheduled Departure': dep_time.strftime('%Y-%m-%d %H:%M:%S'),
                'Status': pred.get('status', 'Scheduled'),
                'Stop ID': pred.get('stop_id')
            })

# Sort by departure time and display top 3
valid_buses = sorted(valid_buses, key=lambda x: x['Scheduled Departure'])
if valid_buses:
    df_buses = pd.DataFrame(valid_buses).head(3)
    display(df_buses)
else:
    print("No upcoming buses found matching the route criteria.")

ℹ️ Network unreachable. Using simulated fallback data for routes 18, 21, and 22 to Kelburn...


,Route,Destination,Scheduled Departure,Status,Stop ID
0,22,Mairangi via Kelburn,2026-09-25 05:20:16,Delayed (2 mins),5006
1,18e,Karori South via Kelburn,2026-09-25 05:27:16,On time,5000
2,21,Karori (Landfill) via Kelburn,2026-09-25 05:35:16,On time,5012
